# PC API — quick test notebook

Test all three endpoints against the running local API.

**Before running:**
1. Start the API in a terminal: `pcenv\Scripts\activate` then `python -m app.main`
2. Keep it running; run the cells below top-to-bottom.
3. `INTERNAL_KEY` below must match `INTERNAL_KEY` in your `.env` (`dev-secret-123`).


## 1. Setup — connect to the API

In [ ]:
import requests, uuid

BASE_URL = "http://127.0.0.1:8000"
INTERNAL_KEY = "dev-secret-123"          # must match INTERNAL_KEY in .env
HEADERS = {"X-Internal-Key": INTERNAL_KEY}

def call(method, path, body=None):
    """Send a request and pretty-print the status + JSON."""
    r = requests.request(method, BASE_URL + path, headers=HEADERS, json=body, timeout=15)
    print(method, path, "->", r.status_code)
    try:
        return r.json()
    except Exception:
        print(r.text); return None

## 2. Health check (no auth needed)

In [ ]:
call("GET", "/healthz")

## 3. Start a session
Creates a fresh user (random UUID) and returns the first 10 cards.

In [ ]:
USER_ID = str(uuid.uuid4())            # a fresh user every run
print("user_id:", USER_ID)

sess = call("POST", f"/api/start_session/{USER_ID}")
SESSION_ID = sess["session_id"]
cards = sess["data"]
print("session_id:", SESSION_ID)
print(f"{len(cards)} cards | rights_left={sess['right_swipes_left']} | products_left={sess['products_left']}")
for c in cards[:5]:
    print("   ", c["product_id"], "-", c["percent_match"], "% match")

## 4. Send one swipe (`/api/suggest`)
Swipe RIGHT on the first card. Most calls return an ack (`data: []`); a refill batch arrives when the queue runs low.

In [ ]:
first = cards[0]["product_id"]
resp = call("POST", "/api/suggest", {
    "session_id": SESSION_ID,
    "ProductInteraction": [{
        "product_id": first, "swipe": "right",
        "details_read": False, "last_image_discovered": 1,
        "bought_in_session": False,
    }],
    "products_swiped": 1,
})
new = resp["data"] if isinstance(resp["data"], list) else resp["data"]
print("type:", resp["type"], "| new cards:", len(new) if isinstance(new, list) else new)
print("total_swipes:", resp["total_swipes"], "| rights_left:", resp["right_swipes_left"])

## 5. Swipe through the whole session
Keeps swiping (mix of right/left) and appending refills until the session ends (rights cap = 8, or products exhausted).

In [ ]:
queue = [c["product_id"] for c in cards]
i = 0
while queue:
    pid = queue.pop(0)
    swipe = "right" if i % 3 == 0 else "left"     # ~1/3 right
    resp = call("POST", "/api/suggest", {
        "session_id": SESSION_ID,
        "ProductInteraction": [{"product_id": pid, "swipe": swipe}],
    })
    i += 1
    if resp["type"] == "ended":
        print("\nSESSION ENDED ->", resp["data"])
        break
    queue += [c["product_id"] for c in resp["data"]]
    print(f"swipe {i}: {swipe:5} | queue={len(queue)} | rights_left={resp['right_swipes_left']} | products_left={resp['products_left']}")

## 6. (Optional) Confirm it persisted to the LOCAL DB
Reads the `pc_*` tables straight from the local Supabase Postgres. Writes are async, so give it a second.

In [ ]:
import time, psycopg2
time.sleep(2)   # let background DB writes flush

c = psycopg2.connect(host="127.0.0.1", port=54422, dbname="postgres",
                     user="postgres", password="postgres")
cur = c.cursor()
for t in ("pc_sessions", "pc_session_items", "user_taste"):
    cur.execute(f"SELECT count(*) FROM public.{t}")
    print(f"{t}: {cur.fetchone()[0]} rows")
cur.execute("SELECT card_position, swipe, percent_match FROM pc_session_items "
            "WHERE session_id=%s ORDER BY card_position LIMIT 12", (SESSION_ID,))
print("\nthis session's items:")
for row in cur.fetchall():
    print("  pos", row[0], "| swipe:", row[1], "| match:", row[2])
c.close()